# <center> Note for Usage </center>
### Just run the first two code blocks in this notebook (install dependencies) and use the last 2 code blocks for inference (run the ner + mapping system) 

If you want to train the model further, the intructions are given in the cells below

# Install all dependencies
Make sure you run these two code blocks given below before using the code

In [33]:
!pip install spacy-transformers
!python -m spacy download en_core_web_md


[notice] A new release of pip is available: 23.1.2 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.1.2 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



                                              0.0/42.8 MB ? eta -:--:--
                                              0.1/42.8 MB 1.3 MB/s eta 0:00:33
                                              0.2/42.8 MB 2.3 MB/s eta 0:00:19
                                              0.3/42.8 MB 2.3 MB/s eta 0:00:19
                                              0.5/42.8 MB 2.4 MB/s eta 0:00:18
                                              0.6/42.8 MB 2.5 MB/s eta 0:00:17
                                              0.7/42.8 MB 2.3 MB/s eta 0:00:19
                                              0.8/42.8 MB 2.5 MB/s eta 0:00:18
                                              0.9/42.8 MB 2.3 MB/s eta 0:00:18
                                              1.0/42.8 MB 2.4 MB/s eta 0:00:18
     -                                        1.1/42.8 MB 2.5 MB/s eta 0:00:17
     -                                        1.2/42.8 MB 2.5 MB/s eta 0:00:17
     -                                        1.4/42.8 MB 

In [ ]:
!pip install transformers sentence-transformers

# SpaCy en-core-web-md pre-trained (For item, loc detection)

# Training Code (Don't run unless you want to train model)

**How to Train The Model**<br>
- To increase the dataset, add more samples to the extended_dataset.json file in the same format and train the NER model for more accurate item and location detection.
- After increasing the dataset, run the code block below
- After the code block is done running, you will see a folder saved in the same directory called "custom_ner_model_md", that is your saved model
- Now you can go to the inference code blocks in the end of the notebook to use your newly trained model!

In [34]:
import spacy
from spacy.training.example import Example
import random
import json

# Load the medium-sized SpaCy model
nlp = spacy.load("en_core_web_md")

# Get the named entity recognizer component
ner = nlp.get_pipe("ner")

# Load the dataset
with open('extended_dataset.json', 'r') as f:
    data = json.load(f)

# Add labels to the NER model
for item in data:
    for label in item['labels']:
        ner.add_label(label[1])

# Convert dataset into SpaCy training format
TRAIN_DATA = []
for item in data:
    entities = []
    for label in item['labels']:
        start = item['text'].index(label[0])
        end = start + len(label[0])
        entities.append((start, end, label[1]))
    TRAIN_DATA.append((item['text'], {"entities": entities}))

# Disable other pipes during training to train only NER
unaffected_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
with nlp.disable_pipes(*unaffected_pipes):
    optimizer = nlp.resume_training()
    for i in range(50):  # Increase number of iterations
        random.shuffle(TRAIN_DATA)
        losses = {}
        for text, annotations in TRAIN_DATA:
            example = Example.from_dict(nlp.make_doc(text), annotations)
            nlp.update([example], losses=losses, drop=0.3)  # Adjust dropout rate
        print(f"Iteration {i}: Losses: {losses}")

# Save the model to disk
model_path = "custom_ner_model_md"
nlp.to_disk(model_path)

print("Model trained and saved successfully.")

Iteration 0: Losses: {'ner': 116.25246766113762}
Iteration 1: Losses: {'ner': 108.09626733578023}
Iteration 2: Losses: {'ner': 77.1234029224735}
Iteration 3: Losses: {'ner': 54.872040003399064}
Iteration 4: Losses: {'ner': 32.582540983920865}
Iteration 5: Losses: {'ner': 23.808150357446827}
Iteration 6: Losses: {'ner': 24.185112950231115}
Iteration 7: Losses: {'ner': 20.549665905762755}
Iteration 8: Losses: {'ner': 9.148728543448863}
Iteration 9: Losses: {'ner': 3.3485938136479114}
Iteration 10: Losses: {'ner': 1.956636929351603}
Iteration 11: Losses: {'ner': 2.9292531150255288}
Iteration 12: Losses: {'ner': 4.1940393280335995}
Iteration 13: Losses: {'ner': 0.5234062462410782}
Iteration 14: Losses: {'ner': 0.15981398265945518}
Iteration 15: Losses: {'ner': 0.520028434837878}
Iteration 16: Losses: {'ner': 0.01417345075255493}
Iteration 17: Losses: {'ner': 0.006438582749102776}
Iteration 18: Losses: {'ner': 0.1951194845677979}
Iteration 19: Losses: {'ner': 0.0025797301762954545}
Iteratio

# Evaluation Metrics

In [35]:
from sklearn.metrics import precision_recall_fscore_support

def evaluate_model(nlp, test_data):
    true_labels = []
    pred_labels = []
    for text, annotations in test_data:
        doc = nlp(text)
        true_ents = annotations["entities"]
        pred_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
        true_labels.extend([label for _, _, label in true_ents])
        pred_labels.extend([label for _, _, label in pred_ents])
    
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='weighted')
    return precision, recall, f1

# test data
test_data = [
    ("Send a box of chocolates to room 101.", {"entities": [(10, 27, "ITEM"), (31, 39, "LOCATION")]}),
    ("Fix the heater in the guest suite.", {"entities": [(9, 15, "ITEM"), (23, 34, "LOCATION")]}),
    ("Deliver fresh fruits to the gym.", {"entities": [(8, 19, "ITEM"), (23, 26, "LOCATION")]}),
    ("Replace the old mattress in the deluxe room.", {"entities": [(12, 24, "ITEM"), (33, 44, "LOCATION")]}),
]

nlp = spacy.load("custom_ner_model_md")

precision, recall, f1 = evaluate_model(nlp, test_data)
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}, F1-score: {f1:.3f}")

Precision: 1.000, Recall: 1.000, F1-score: 1.000


# Extensive test samples

In [36]:
import spacy

nlp = spacy.load("custom_ner_model_md")

# Test data
test_data = [
    "Send a box of chocolates to room 101.",
    "Fix the heater in the guest suite.",
    "Deliver fresh fruits to the gym.",
    "Replace the old mattress in the deluxe room.",
    "Send a bouquet of flowers to the bridal suite.",
    "Repair the broken window in the dining hall.",
    "Deliver a bottle of champagne to the honeymoon suite.",
    "Fix the leaky faucet in the second-floor bathroom.",
    "Send a set of golf clubs to the sports center.",
    "Deliver dinner to the poolside cabana.",
    "Send a first aid kit to the front desk.",
    "Repair the projector in the training room.",
    "Deliver a new TV to the lounge area.",
    "Fix the sound system in the banquet hall.",
    "Send a pack of playing cards to the game room.",
    "Deliver a welcome package to room 312.",
    "Send a case of bottled water to the conference center.",
    "Repair the printer in the business lounge.",
    "Send extra linens to the master suite.",
    "Fix the thermostat in the family room."
]

for text in test_data:
    doc = nlp(text)
    print(f"Text: {text}")
    for ent in doc.ents:
        print(f" - {ent.text}: {ent.label_}")

Text: Send a box of chocolates to room 101.
 - box of chocolates: ITEM
 - room 101: LOCATION
Text: Fix the heater in the guest suite.
 - heater: ITEM
 - guest suite: LOCATION
Text: Deliver fresh fruits to the gym.
 - fresh fruits: ITEM
 - gym: LOCATION
Text: Replace the old mattress in the deluxe room.
 - old mattress: ITEM
 - deluxe room: LOCATION
Text: Send a bouquet of flowers to the bridal suite.
 - bouquet of flowers: ITEM
 - bridal suite: LOCATION
Text: Repair the broken window in the dining hall.
 - broken window: ITEM
 - dining hall: LOCATION
Text: Deliver a bottle of champagne to the honeymoon suite.
 - bottle of champagne: ITEM
 - honeymoon suite: LOCATION
Text: Fix the leaky faucet in the second-floor bathroom.
 - leaky faucet: ITEM
 - second-floor bathroom: LOCATION
Text: Send a set of golf clubs to the sports center.
 - set of golf clubs: ITEM
 - sports center: LOCATION
Text: Deliver dinner to the poolside cabana.
 - dinner: ITEM
 - poolside cabana: LOCATION
Text: Send a f

# Test samples with missing information

In [37]:
import spacy

nlp = spacy.load("custom_ner_model_md")

# Test the model with sentences where item or location is missing
test_data_missing = [
    "Send to room 101.",
    "Fix the heater.",
    "Deliver fresh fruits.",
    "Replace the old mattress.",
    "Send a bouquet of flowers to the bridal suite.",
    "Repair the broken window.",
    "Deliver a bottle of champagne.",
    "Fix the leaky faucet in the second-floor bathroom.",
    "Send a set of golf clubs.",
    "Deliver dinner to the poolside cabana.",
    "Send a first aid kit to the front desk.",
    "Repair the projector.",
    "Deliver a new TV to the lounge area.",
    "Fix the sound system.",
    "Send a pack of playing cards to the game room.",
    "Deliver a welcome package.",
    "Send a case of bottled water.",
    "Repair the printer in the business lounge.",
    "Send extra linens.",
    "Fix the thermostat."
]

for text in test_data_missing:
    doc = nlp(text)
    print(f"Text: {text}")
    for ent in doc.ents:
        print(f" - {ent.text}: {ent.label_}")

Text: Send to room 101.
 - room 101: LOCATION
Text: Fix the heater.
 - heater: LOCATION
Text: Deliver fresh fruits.
 - fresh fruits: ITEM
Text: Replace the old mattress.
 - old mattress: ITEM
Text: Send a bouquet of flowers to the bridal suite.
 - bouquet of flowers: ITEM
 - bridal suite: LOCATION
Text: Repair the broken window.
 - broken window: ITEM
Text: Deliver a bottle of champagne.
 - bottle of champagne: ITEM
Text: Fix the leaky faucet in the second-floor bathroom.
 - leaky faucet: ITEM
 - second-floor bathroom: LOCATION
Text: Send a set of golf clubs.
 - set of golf clubs: ITEM
Text: Deliver dinner to the poolside cabana.
 - dinner: ITEM
 - poolside cabana: LOCATION
Text: Send a first aid kit to the front desk.
 - first aid kit: LOCATION
 - front desk: LOCATION
Text: Repair the projector.
 - projector: ITEM
Text: Deliver a new TV to the lounge area.
 - TV: ITEM
 - lounge area: LOCATION
Text: Fix the sound system.
 - sound system: ITEM
Text: Send a pack of playing cards to the g

# Input the text you want and get the output
You can use this code block to test the NER model alone

In [38]:
import spacy

nlp = spacy.load("custom_ner_model_md")

text = input("Enter the text: ")

doc = nlp(text)
print(f"Text: {text}")
for ent in doc.ents:
    print(f" - {ent.text}: {ent.label_}")

Text: Set up the game console in the entertainment area.
 - game console: ITEM
 - entertainment area: LOCATION


# <center>Mapping Transformer Model</center>

In [39]:
# List of items and locations (standardized lists)
items = ["ac units", "alarm clock", "amenities", "amenities tray", "amenity", "artwork", "ash tray", 
         "bathmat", "bathroom light", "bath rug", "bath towel", "bed", "bed frame", "bedroom light", 
         "bed runner", "blanket", "blinds", "blinds broken", "body soap", "bottled water", "breakfast order form", 
         "buffet area", "carpet - torn", "ceiling", "ceiling - leak", "ceiling light", "chair", "chemical reading", 
         "cleaning chemicals", "cleaning rags", "closet door", "closet light", "coffee maker", "coffee mug", 
         "coffee pods - decaf", "coffee pods - regular", "coffee set up", "coffee table books", "comb", "computer", 
         "conditioner", "corkscrew", "couch - cushion", "crib", "crib sheets", "crown molding", "deodorant - female", 
         "deodorant - male", "departure clean", "desk", "desk lamp", "desk light", "dishes", "dishwasher", 
         "dishwasher filters", "distribution board", "dnd sign", "door", "door - caulk", "door - handle broken", 
         "door - lock", "door - loose/not closing", "door - noisy", "door - scratched", "door - sliding glass", 
         "dp-item", "dp-towels", "dresser", "elevator doors", "elevator light", "entry light", "ethernet cord", 
         "extension cord", "fan", "faucets", "feminine products", "flat surfaces", "floor", "floor - carpet", 
         "floor - hardwood", "floor - marble", "floor - pavement", "floor - tile", "floor - wood", "flowers", 
         "gift shop item", "glasses", "guest elevator", "hair dryer", "hairspray", "hallway light", 
         "hallway - wall damaged", "hand soap", "hand towel", "hanger - clips", "hanger - female", "hanger - male", 
         "housekeeping job", "humidifier", "ice", "in room ipad", "inspect cabinets", "inspect dishwasher", 
         "inspect door handle", "inspect refrigerator", "inspect the exterior of the guest room door", "ipad -", 
         "iron", "is the lock functional", "kitchen appliances", "kleenex", "lamp", "laptop", "lap top cable", 
         "laundry", "laundry bag", "laundry dryer", "laundry slip", "leaking tap", "light bulbs", "lightout - entryway", 
         "light switch", "linen - strip room", "lint roller", "lost and found item", "lotion", "luggage", 
         "luggage rack", "make up mirror", "makeup remover wipes", "marble - cracked", "matches", "mattress", 
         "mattress protector", "michaels towel", "microwave", "milkshake", "minibar", "minibar items", "mini fridge", 
         "mirror", "mirror - cracked", "mouthwash", "news clips", "news journals", "newspaper", "night light", 
         "optii journal", "package", "paper", "pen", "pencil", "phone charger", "pillow", "pillow - feather", 
         "pillow (firm)", "pillow - foam", "plumbing", "pm room", "pool equipment", "pool towel", "q tips", "razor", 
         "reading light", "refrigerator", "remote batteries", "remote control", "remove rollaway", "restrooms", "robe", 
         "rollaway bed", "room number", "room service menu", "room service table", "room service tray", "rotate mattress", 
         "rs tray", "safe", "safe - will not open", "safe - will not shut", "security latch", "sewing kit", "shampoo", 
         "shower", "shower cap", "shower - curtain rod", "shower gel", "shower - glass door", "shower - grab bar", 
         "shower - grout", "shower - handle", "showerhead", "shower head", "shower - leaking", "shower - no cold water", 
         "shower - no hot water", "shower - switch", "shower - tile", "silverware", "sink", "sink - caulking", 
         "sink - faucet", "sink - handle", "sink - not draining", "slippers", "smoothie", "soap dish", "sparkling water", 
         "speaker", "tea bags", "telephone", "thermostat", "thiagos hippie shirt", "tissue box", "tissues", "toilet", 
         "toilet - base broken", "toilet - clogged", "toilet - handle broken", "toilet - not flushing", "toilet paper", 
         "toilet - seat loose", "tony santa", "toothbrush", "toothpaste", "touch up stain in/out", "towel", "towels", 
         "trash", "trash can liner", "travel adapter", "travel kit", "t-shirt", "tub", "tub - not draining", "tv", 
         "tv - no picture", "tv - no sound", "tv reception", "tv - will not turn off", "tv - will not turn on", "vacuum", 
         "vanity", "vanity - cracked", "vanity kit", "vents (clean)", "video games", "wall marks", "wallpaper", 
         "washcloth", "water glass", "water heater", "window", "window - curtain", "window - curtain broken", "zztop"]

locations = ["conference room 203", "level 1000", "rm 201", "rm 202", "rm 203", "rm 204", "rm 205", "rm 101010", 
             "rm 200001", "s a", "f 301", "entertainment area", "rm 301", "rm 302", "rm 303", "rm 304", "rm 305", 
             "rm 30000", "rm 401", "rm 402", "rm 404", "rm 405", "rm 406", "level 4 linen storage", "rm 501", 
             "rm 502", "rm 503", "rm 504", "rm 505", "rm 507", "room 501a", "level 5 linen storage", "rm 601", 
             "rm 602", "rm 603", "rm 604", "rm 605", "rm 606", "rm 607", "rm 701", "rm 702", "rm 703", "rm 704", 
             "rm 707", "rm 801", "rm 803", "rm 805", "rm 807", "level 8 linen storage", "rm 901", "rm 902", 
             "rm 903", "rm 904", "rm 905", "rm 906", "rm 907", "bar", "sunshine room", "rm 1006", "rm 1007", 
             "rm 1020", "rm 10006", "rm 100001", "rm 12", "rm 1001", "rm 1004", "rm 1005", "rm 1101", "rm 1102", 
             "rm 1103", "rm 1104", "rm 1105", "rm 1106", "rm 1107", "storage 2", "level 11 linen storage", "rm 1201", 
             "rm 1202", "rm 1203", "rm 1204", "rm 1205", "rm 1206", "rm 1207", "rm 1301", "rm 1304", "rm 1305", 
             "rm 1306", "rm 1307", "rm 1401", "rm 1402", "rm 1403", "rm 1406", "level 14 linen storage", "rm 1501", 
             "rm 1502", "rm 1503", "rm 1504", "rm 1505", "rm 1506", "rm 1507", "rm 1601", "rm 1603", "rm 1604", 
             "rm 1605", "rm 1607", "rm 1701", "rm 1702", "rm 1703", "rm 1704", "rm 1706", "rm 1707", "rm 1802", 
             "rm 1804", "rm 1805", "rm 1806", "rm 1807", "rm 1901", "rm 1902", "rm 1904", "rm 1905", "rm 1906", 
             "rm 1907", "level 19 linen storage", "rm 2001", "rm 2002", "rm 2003", "rm 2006", "level 20 linen storage", 
             "rm 2101", "rm 2102", "rm 2103", "rm 2104", "rm 2105", "rm 2106", "rm 2107", "rm 2201", "rm 2202", 
             "rm 2203", "rm 2205", "rm 2206", "rm 2207", "rm 10000", "rm 2301", "rm 2302", "rm 2303", "rm 2304", 
             "rm 2307", "level 23 linen storage", "rm 2401", "rm 2402", "rm 2403", "rm 2404", "rm 2405", "rm 2406", 
             "rm 2407", "rm 2501", "rm 2502", "rm 2504", "rm 2505", "rm 2506", "rm 2507", "rm 2601", "rm 2602", 
             "rm 2604", "rm 2606", "rm 2607", "level 26 linen storage", "level 27 disabled bathroom", 
             "level 27 female bathroom", "level 27 male bathroom", "conference room 1", "conference room 2", "gymnasium", 
             "outside balcony", "residents lounge", "rm 2801", "rm 2802", "rm 2803", "rm 2804", "rm 2901", "rm 2902", 
             "rm 2903", "rm 2904", "rm 3001", "rm 3004", "level 30 linen storage", "rm 3101", "rm 3102", "rm 3103", 
             "rm 3104", "rm 3201", "rm 3202", "rm 3203", "rm 3204", "rm 3301", "rm 3302", "rm 3303", "rm 3304", 
             "level 33 linen storage", "rm 3402", "rm 3403", "rm 3404", "rm 3501", "rm 3502", "rm 3503", "rm 3504", 
             "rm 3602", "rm 3603", "rm 3604", "rm 3701", "rm 3702", "rm 3703", "rm 3704", "level 37 linen storage", 
             "rm 3901", "rm 3902", "rm 3903", "level 40 bathroom", "level 40 bbqs", "level 40 gardens", "rm 4001", 
             "rm 4002", "rm 4003", "rm 4004", "foxtel boxes", "hot water system", "lift motor room", "basement 101", 
             "b1 storage mgmt", "pool plantroom", "south building", "rm 20000", "b2 storage mgmt", "chemical room", 
             "garbage room", "housekeeping linen room", "housekeeping office", "rm 101", "maintenance manager office v2", 
             "maintenance workshop", "staffroom", "storage bc/paint", "storage owners", "b3 housekeeping storage closet", 
             "rm 100000", "service room", "level 100", "retail bathrooms", "back office", "fire panel room", "bbqs", 
             "pool chemical room", "sauna", "spa pool", "steam room", "swimming pool", "lift 1", "lift 2", "lift 3", 
             "lobby", "f 1000", "anthonys 7667 - 12345", "b1", "b2", "b3", "fire stairwell east", "fire stairwell west", 
             "level 2", "level 3", "level 4", "level 5", "level 6", "level 7", "level 8", "level 9", "level 10", 
             "level 11", "level 12", "level 13", "level 14", "level 15", "level 16", "level 17", "level 18", "level 19", 
             "level 20", "level 21", "level 22", "level 23", "level 24", "level 25", "level 26", "level 27", "level 28", 
             "level 29", "level 30", "level 31", "level 32", "level 33", "level 34", "level 35", "level 36", "level 37", 
             "level 38", "level 39", "level 40", "level 41", "level 42", "ground level", "gas heater for pool", 
             "outside gardens", "outside patio", "retail area", "rhapsody beachside restaurant", "great optii hotel", 
             "the great optii hotel #1"]

# <center>Complete System: NER + Mapper</center>

## paraphrase-mpnet-base-v2

In [40]:
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
import re

# large pre-trained model paraphrase-mpnet-base-v2
model = SentenceTransformer('paraphrase-mpnet-base-v2')

def map_to_standardized(sentence, detected_item, detected_location, items, locations):
    # Return "None" if the detected item or location is "None"
    if detected_item == "None":
        best_match_item = "None"
    else:
        item_context_embedding = model.encode(f"{detected_item} {sentence}", convert_to_tensor=True)
        item_embeddings = model.encode(items, convert_to_tensor=True)
        item_similarities = util.pytorch_cos_sim(item_context_embedding, item_embeddings).flatten()
        item_similarities = item_similarities.cpu().numpy()
        best_match_item_idx = np.argmax(item_similarities)
        best_match_item = items[best_match_item_idx]

    if detected_location == "None":
        best_match_location = "None"
    else:
        location_context_embedding = model.encode(f"{detected_location} {sentence}", convert_to_tensor=True)
        location_embeddings = model.encode(locations, convert_to_tensor=True)
        location_similarities = util.pytorch_cos_sim(location_context_embedding, location_embeddings).flatten()
        location_similarities = location_similarities.cpu().numpy()

        # Custom rule to handle "room" with a number
        detected_location_number = re.findall(r'room (\d+)', detected_location, re.IGNORECASE)
        if detected_location_number:
            detected_location_number = detected_location_number[0]
            location_similarity_scores = []
            for location in locations:
                location_number = re.findall(r'rm (\d+)', location, re.IGNORECASE)
                if location_number and location_number[0] == detected_location_number:
                    location_similarity_scores.append(1.0)
                else:
                    location_similarity_scores.append(0.0)
            location_similarity_scores = np.array(location_similarity_scores)
            best_match_location_idx = np.argmax(location_similarity_scores)
        else:
            best_match_location_idx = np.argmax(location_similarities)

        best_match_location = locations[best_match_location_idx]

    return best_match_item, best_match_location

c:\Users\Fatima Azfar\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### results of paraphrase-mpnet-base-v2

In [41]:
import spacy

nlp = spacy.load("custom_ner_model_md")

test_sentences = [
    # Item: air conditioner (ac units)
    "Check the air conditioner in room 305.",
    
    # Item: wake-up clock (alarm clock)
    "Replace the wake-up clock in room 202.",
    
    # Item: amenities basket (amenities tray)
    "Restock the amenities basket in the deluxe suite.",
    
    # Item: bath mat (bathmat)
    "Change the bath mat in the second-floor bathroom.",
    
    # Item: bedroom lamp (bedroom light)
    "Fix the bedroom lamp in the master suite.",
    
    # Item: comforter (blanket)
    "Replace the comforter in room 1101.",
    
    # Item: window shades (blinds)
    "Repair the window shades in the living area.",
    
    # Item: hand soap (body soap)
    "Restock the hand soap in the guest bathroom.",
    
    # Item: mineral water (bottled water)
    "Send a case of mineral water to the gym.",
    
    # Item: carpet (floor - carpet)
    "Clean the carpet in the conference room.",
    
    # Item: TV set (tv)
    "Install a new TV set in the lounge.",
    
    # Item: game console (video games)
    "Set up the game console in the entertainment area.",
    
    # Location: gymnasium (gym)
    "Deliver fresh towels to the gymnasium.",
    
    # Location: poolside (swimming pool)
    "Send drinks to the poolside cabana.",
    
    # Location: lounge (residents lounge)
    "Replace the magazines in the lounge.",
    
    # Location: storage area (level 4 linen storage)
    "Check the inventory in the storage area on level 4.",
    
    # Location: maintenance room (maintenance workshop)
    "Repair the tools in the maintenance room.",
    
    # Location: front office (front desk)
    "Deliver the package to the front office.",
    
    # Location: main hall (lobby)
    "Set up the decorations in the main hall.",
    
    # Location: spa (spa pool)
    "Check the temperature of the water in the spa."
]

detected_item = "None"
detected_location = "None"
for text in test_sentences:
    doc = nlp(text)
    print(f"Text: {text}")
    for ent in doc.ents:
        print(f" - {ent.text}: {ent.label_}")
        if ent.label_ == "ITEM":
            detected_item = ent.text
        elif ent.label_ == "LOCATION":
            detected_location = ent.text
        else:
            detected_item = "None"
            detected_location = "None"
        best_item, best_location = map_to_standardized(text, detected_item, detected_location, items, locations)
    print(f" - Mapped Item: {best_item}")
    print(f" - Mapped Location: {best_location}")

Text: Check the air conditioner in room 305.
 - air conditioner: ITEM
 - room 305: LOCATION
 - Mapped Item: conditioner
 - Mapped Location: rm 305
Text: Replace the wake-up clock in room 202.
 - wake-up clock: ITEM
 - room 202: LOCATION
 - Mapped Item: alarm clock
 - Mapped Location: rm 202
Text: Restock the amenities basket in the deluxe suite.
 - amenities basket: ITEM
 - deluxe suite: LOCATION
 - Mapped Item: amenities tray
 - Mapped Location: service room
Text: Change the bath mat in the second-floor bathroom.
 - bath mat: ITEM
 - second-floor bathroom: LOCATION
 - Mapped Item: bathmat
 - Mapped Location: level 27 female bathroom
Text: Fix the bedroom lamp in the master suite.
 - bedroom lamp: ITEM
 - master suite: LOCATION
 - Mapped Item: bedroom light
 - Mapped Location: housekeeping linen room
Text: Replace the comforter in room 1101.
 - comforter: ITEM
 - room 1101: LOCATION
 - Mapped Item: linen - strip room
 - Mapped Location: rm 1101
Text: Repair the window shades in the liv

## Testing on more data

In [44]:
test_sentences = [
    # Synonyms for items
    "Check the air conditioning unit in room 305.",  # air conditioner
    "Replace the alarm clock in room 202.",  # wake-up clock
    "Restock the amenities tray in the deluxe suite.",  # amenities basket
    "Change the bath rug in the second-floor bathroom.",  # bath mat
    "Fix the bedroom light in the master suite.",  # bedroom lamp
    "Replace the blanket in room 1101.",  # comforter
    "Repair the blinds in the living area.",  # window shades
    "Restock the body soap in the guest bathroom.",  # hand soap
    "Send a case of bottled water to the gym.",  # mineral water
    "Clean the floor - carpet in the conference room.",  # carpet
    "Install a new television in the lounge.",  # TV set
    "Set up the video games in the entertainment area.",  # game console

    # Synonyms for locations
    "Deliver fresh towels to the fitness center.",  # gymnasium
    "Send drinks to the swimming pool area.",  # poolside
    "Replace the magazines in the residents lounge.",  # lounge
    "Check the inventory in the linen storage on level 4.",  # storage area
    "Repair the tools in the maintenance workshop.",  # maintenance room
    "Deliver the package to the front desk.",  # front office
    "Set up the decorations in the lobby.",  # main hall
    "Check the water temperature in the spa pool."  # spa
]

detected_item = "None"
detected_location = "None"
for text in test_sentences:
    doc = nlp(text)
    print(f"Text: {text}")
    for ent in doc.ents:
        print(f" - {ent.text}: {ent.label_}")
        if ent.label_ == "ITEM":
            detected_item = ent.text
        elif ent.label_ == "LOCATION":
            detected_location = ent.text
        else:
            detected_item = "None"
            detected_location = "None"
        best_item, best_location = map_to_standardized(text, detected_item, detected_location, items, locations)
    print(f" - Mapped Item: {best_item}")
    print(f" - Mapped Location: {best_location}")

Text: Check the air conditioning unit in room 305.
 - air conditioning unit: ITEM
 - room 305: LOCATION
 - Mapped Item: conditioner
 - Mapped Location: rm 305
Text: Replace the alarm clock in room 202.
 - alarm clock: ITEM
 - room 202: LOCATION
 - Mapped Item: alarm clock
 - Mapped Location: rm 202
Text: Restock the amenities tray in the deluxe suite.
 - amenities tray: ITEM
 - deluxe suite: LOCATION
 - Mapped Item: amenities tray
 - Mapped Location: service room
Text: Change the bath rug in the second-floor bathroom.
 - bath rug: ITEM
 - second-floor bathroom: LOCATION
 - Mapped Item: bath rug
 - Mapped Location: level 27 female bathroom
Text: Fix the bedroom light in the master suite.
 - bedroom light: ITEM
 - master suite: LOCATION
 - Mapped Item: bedroom light
 - Mapped Location: housekeeping linen room
Text: Replace the blanket in room 1101.
 - blanket: ITEM
 - room 1101: LOCATION
 - Mapped Item: blanket
 - Mapped Location: rm 1101
Text: Repair the blinds in the living area.
 - bl

# <center>Inference Code</center>
### <center>You can run the  code blocks below freely</center>
<center>Use the 2 code blocks below to use the model on your text samples!</center>

In [3]:
import spacy
from sentence_transformers import SentenceTransformer, util
import numpy as np
import re

nlp = spacy.load("custom_ner_model_md")

# large pre-trained model paraphrase-mpnet-base-v2
model = SentenceTransformer('paraphrase-mpnet-base-v2')

def map_to_standardized(sentence, detected_item, detected_location, items, locations):
    # Return "None" if the detected item or location is "None"
    if detected_item == "None":
        best_match_item = "None"
    else:
        item_context_embedding = model.encode(f"{detected_item} {sentence}", convert_to_tensor=True)
        item_embeddings = model.encode(items, convert_to_tensor=True)
        item_similarities = util.pytorch_cos_sim(item_context_embedding, item_embeddings).flatten()
        item_similarities = item_similarities.cpu().numpy()
        best_match_item_idx = np.argmax(item_similarities)
        best_match_item = items[best_match_item_idx]

    if detected_location == "None":
        best_match_location = "None"
    else:
        location_context_embedding = model.encode(f"{detected_location} {sentence}", convert_to_tensor=True)
        location_embeddings = model.encode(locations, convert_to_tensor=True)
        location_similarities = util.pytorch_cos_sim(location_context_embedding, location_embeddings).flatten()
        location_similarities = location_similarities.cpu().numpy()

        # Custom rule to handle "room" with a number
        detected_location_number = re.findall(r'room (\d+)', detected_location, re.IGNORECASE)
        if detected_location_number:
            detected_location_number = detected_location_number[0]
            location_similarity_scores = []
            for location in locations:
                location_number = re.findall(r'rm (\d+)', location, re.IGNORECASE)
                if location_number and location_number[0] == detected_location_number:
                    location_similarity_scores.append(1.0)
                else:
                    location_similarity_scores.append(0.0)
            location_similarity_scores = np.array(location_similarity_scores)
            best_match_location_idx = np.argmax(location_similarity_scores)
        else:
            best_match_location_idx = np.argmax(location_similarities)

        best_match_location = locations[best_match_location_idx]

    return best_match_item, best_match_location

# List of items and locations (standardized lists)
items = ["ac units", "alarm clock", "amenities", "amenities tray", "amenity", "artwork", "ash tray", 
         "bathmat", "bathroom light", "bath rug", "bath towel", "bed", "bed frame", "bedroom light", 
         "bed runner", "blanket", "blinds", "blinds broken", "body soap", "bottled water", "breakfast order form", 
         "buffet area", "carpet - torn", "ceiling", "ceiling - leak", "ceiling light", "chair", "chemical reading", 
         "cleaning chemicals", "cleaning rags", "closet door", "closet light", "coffee maker", "coffee mug", 
         "coffee pods - decaf", "coffee pods - regular", "coffee set up", "coffee table books", "comb", "computer", 
         "conditioner", "corkscrew", "couch - cushion", "crib", "crib sheets", "crown molding", "deodorant - female", 
         "deodorant - male", "departure clean", "desk", "desk lamp", "desk light", "dishes", "dishwasher", 
         "dishwasher filters", "distribution board", "dnd sign", "door", "door - caulk", "door - handle broken", 
         "door - lock", "door - loose/not closing", "door - noisy", "door - scratched", "door - sliding glass", 
         "dp-item", "dp-towels", "dresser", "elevator doors", "elevator light", "entry light", "ethernet cord", 
         "extension cord", "fan", "faucets", "feminine products", "flat surfaces", "floor", "floor - carpet", 
         "floor - hardwood", "floor - marble", "floor - pavement", "floor - tile", "floor - wood", "flowers", 
         "gift shop item", "glasses", "guest elevator", "hair dryer", "hairspray", "hallway light", 
         "hallway - wall damaged", "hand soap", "hand towel", "hanger - clips", "hanger - female", "hanger - male", 
         "housekeeping job", "humidifier", "ice", "in room ipad", "inspect cabinets", "inspect dishwasher", 
         "inspect door handle", "inspect refrigerator", "inspect the exterior of the guest room door", "ipad -", 
         "iron", "is the lock functional", "kitchen appliances", "kleenex", "lamp", "laptop", "lap top cable", 
         "laundry", "laundry bag", "laundry dryer", "laundry slip", "leaking tap", "light bulbs", "lightout - entryway", 
         "light switch", "linen - strip room", "lint roller", "lost and found item", "lotion", "luggage", 
         "luggage rack", "make up mirror", "makeup remover wipes", "marble - cracked", "matches", "mattress", 
         "mattress protector", "michaels towel", "microwave", "milkshake", "minibar", "minibar items", "mini fridge", 
         "mirror", "mirror - cracked", "mouthwash", "news clips", "news journals", "newspaper", "night light", 
         "optii journal", "package", "paper", "pen", "pencil", "phone charger", "pillow", "pillow - feather", 
         "pillow (firm)", "pillow - foam", "plumbing", "pm room", "pool equipment", "pool towel", "q tips", "razor", 
         "reading light", "refrigerator", "remote batteries", "remote control", "remove rollaway", "restrooms", "robe", 
         "rollaway bed", "room number", "room service menu", "room service table", "room service tray", "rotate mattress", 
         "rs tray", "safe", "safe - will not open", "safe - will not shut", "security latch", "sewing kit", "shampoo", 
         "shower", "shower cap", "shower - curtain rod", "shower gel", "shower - glass door", "shower - grab bar", 
         "shower - grout", "shower - handle", "showerhead", "shower head", "shower - leaking", "shower - no cold water", 
         "shower - no hot water", "shower - switch", "shower - tile", "silverware", "sink", "sink - caulking", 
         "sink - faucet", "sink - handle", "sink - not draining", "slippers", "smoothie", "soap dish", "sparkling water", 
         "speaker", "tea bags", "telephone", "thermostat", "thiagos hippie shirt", "tissue box", "tissues", "toilet", 
         "toilet - base broken", "toilet - clogged", "toilet - handle broken", "toilet - not flushing", "toilet paper", 
         "toilet - seat loose", "tony santa", "toothbrush", "toothpaste", "touch up stain in/out", "towel", "towels", 
         "trash", "trash can liner", "travel adapter", "travel kit", "t-shirt", "tub", "tub - not draining", "tv", 
         "tv - no picture", "tv - no sound", "tv reception", "tv - will not turn off", "tv - will not turn on", "vacuum", 
         "vanity", "vanity - cracked", "vanity kit", "vents (clean)", "video games", "wall marks", "wallpaper", 
         "washcloth", "water glass", "water heater", "window", "window - curtain", "window - curtain broken", "zztop"]

locations = ["conference room 203", "level 1000", "rm 201", "rm 202", "rm 203", "rm 204", "rm 205", "rm 101010", 
             "rm 200001", "s a", "f 301", "entertainment area", "rm 301", "rm 302", "rm 303", "rm 304", "rm 305", 
             "rm 30000", "rm 401", "rm 402", "rm 404", "rm 405", "rm 406", "level 4 linen storage", "rm 501", 
             "rm 502", "rm 503", "rm 504", "rm 505", "rm 507", "room 501a", "level 5 linen storage", "rm 601", 
             "rm 602", "rm 603", "rm 604", "rm 605", "rm 606", "rm 607", "rm 701", "rm 702", "rm 703", "rm 704", 
             "rm 707", "rm 801", "rm 803", "rm 805", "rm 807", "level 8 linen storage", "rm 901", "rm 902", 
             "rm 903", "rm 904", "rm 905", "rm 906", "rm 907", "bar", "sunshine room", "rm 1006", "rm 1007", 
             "rm 1020", "rm 10006", "rm 100001", "rm 12", "rm 1001", "rm 1004", "rm 1005", "rm 1101", "rm 1102", 
             "rm 1103", "rm 1104", "rm 1105", "rm 1106", "rm 1107", "storage 2", "level 11 linen storage", "rm 1201", 
             "rm 1202", "rm 1203", "rm 1204", "rm 1205", "rm 1206", "rm 1207", "rm 1301", "rm 1304", "rm 1305", 
             "rm 1306", "rm 1307", "rm 1401", "rm 1402", "rm 1403", "rm 1406", "level 14 linen storage", "rm 1501", 
             "rm 1502", "rm 1503", "rm 1504", "rm 1505", "rm 1506", "rm 1507", "rm 1601", "rm 1603", "rm 1604", 
             "rm 1605", "rm 1607", "rm 1701", "rm 1702", "rm 1703", "rm 1704", "rm 1706", "rm 1707", "rm 1802", 
             "rm 1804", "rm 1805", "rm 1806", "rm 1807", "rm 1901", "rm 1902", "rm 1904", "rm 1905", "rm 1906", 
             "rm 1907", "level 19 linen storage", "rm 2001", "rm 2002", "rm 2003", "rm 2006", "level 20 linen storage", 
             "rm 2101", "rm 2102", "rm 2103", "rm 2104", "rm 2105", "rm 2106", "rm 2107", "rm 2201", "rm 2202", 
             "rm 2203", "rm 2205", "rm 2206", "rm 2207", "rm 10000", "rm 2301", "rm 2302", "rm 2303", "rm 2304", 
             "rm 2307", "level 23 linen storage", "rm 2401", "rm 2402", "rm 2403", "rm 2404", "rm 2405", "rm 2406", 
             "rm 2407", "rm 2501", "rm 2502", "rm 2504", "rm 2505", "rm 2506", "rm 2507", "rm 2601", "rm 2602", 
             "rm 2604", "rm 2606", "rm 2607", "level 26 linen storage", "level 27 disabled bathroom", 
             "level 27 female bathroom", "level 27 male bathroom", "conference room 1", "conference room 2", "gymnasium", 
             "outside balcony", "residents lounge", "rm 2801", "rm 2802", "rm 2803", "rm 2804", "rm 2901", "rm 2902", 
             "rm 2903", "rm 2904", "rm 3001", "rm 3004", "level 30 linen storage", "rm 3101", "rm 3102", "rm 3103", 
             "rm 3104", "rm 3201", "rm 3202", "rm 3203", "rm 3204", "rm 3301", "rm 3302", "rm 3303", "rm 3304", 
             "level 33 linen storage", "rm 3402", "rm 3403", "rm 3404", "rm 3501", "rm 3502", "rm 3503", "rm 3504", 
             "rm 3602", "rm 3603", "rm 3604", "rm 3701", "rm 3702", "rm 3703", "rm 3704", "level 37 linen storage", 
             "rm 3901", "rm 3902", "rm 3903", "level 40 bathroom", "level 40 bbqs", "level 40 gardens", "rm 4001", 
             "rm 4002", "rm 4003", "rm 4004", "foxtel boxes", "hot water system", "lift motor room", "basement 101", 
             "b1 storage mgmt", "pool plantroom", "south building", "rm 20000", "b2 storage mgmt", "chemical room", 
             "garbage room", "housekeeping linen room", "housekeeping office", "rm 101", "maintenance manager office v2", 
             "maintenance workshop", "staffroom", "storage bc/paint", "storage owners", "b3 housekeeping storage closet", 
             "rm 100000", "service room", "level 100", "retail bathrooms", "back office", "fire panel room", "bbqs", 
             "pool chemical room", "sauna", "spa pool", "steam room", "swimming pool", "lift 1", "lift 2", "lift 3", 
             "lobby", "f 1000", "anthonys 7667 - 12345", "b1", "b2", "b3", "fire stairwell east", "fire stairwell west", 
             "level 2", "level 3", "level 4", "level 5", "level 6", "level 7", "level 8", "level 9", "level 10", 
             "level 11", "level 12", "level 13", "level 14", "level 15", "level 16", "level 17", "level 18", "level 19", 
             "level 20", "level 21", "level 22", "level 23", "level 24", "level 25", "level 26", "level 27", "level 28", 
             "level 29", "level 30", "level 31", "level 32", "level 33", "level 34", "level 35", "level 36", "level 37", 
             "level 38", "level 39", "level 40", "level 41", "level 42", "ground level", "gas heater for pool", 
             "outside gardens", "outside patio", "retail area", "rhapsody beachside restaurant", "great optii hotel", 
             "the great optii hotel #1"]

## Input text and get the mapped items and locations

In [4]:
text = input("Enter the text: ")

detected_item = "None"
detected_location = "None"
doc = nlp(text)
print(f"Text: {text}")
for ent in doc.ents:
    print(f" - {ent.text}: {ent.label_}")
    if ent.label_ == "ITEM":
        detected_item = ent.text
    elif ent.label_ == "LOCATION":
        detected_location = ent.text
    best_item, best_location = map_to_standardized(text, detected_item, detected_location, items, locations)
print(f" - Mapped Item: {best_item}")
print(f" - Mapped Location: {best_location}")

Text: Deliver an ipad to the room 101
 - ipad: ITEM
 - room 101: LOCATION
 - Mapped Item: in room ipad
 - Mapped Location: rm 101
